# Hather: teacher Run A (write the Arabic data)

The teacher (abliterated Qwen 3.8 27B) runs on Kaggle's two T4 GPUs through llama.cpp and works through `runA_jobs.jsonl`:
1,500 translations + 300 generation jobs (5 messages each) = **3,000 Arabic messages**.

**Before running**
1. Settings → Accelerator **GPU T4 x2**, Internet **on**.
2. Add Input → your dataset that contains `runA_jobs.jsonl`.
3. Optional, from the 2nd session on: add the `llamacpp-t4` dataset (skips the 15-20 min build) and the previous version's output (so finished jobs are skipped).

**Best way to run:** *Save Version → Save & Run All*. It runs in the background for up to 12 hours, so you can close the browser.
Every answer is saved the moment it arrives, so a timeout never loses work.

In [ ]:
# 1. Settings
import glob, os, json, time, re, subprocess, shutil

# Teacher: abliterated Qwen 3.8 27B (refusals removed, so it writes scam examples without pushing back)
MODEL_REPO = "huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF"
MODEL_FILE = "Huihui-Qwen3.8-27B-abliterated-UD-DW-Q4_K_M.gguf"   # 16.55 GB, single file
# Original model (not abliterated):
# MODEL_REPO = "unsloth/Qwen3.8-27B-GGUF"; MODEL_FILE = "Qwen3.8-27B-UD-Q4_K_M.gguf"
PARALLEL   = 4                                       # requests at once
CTX_PER_SLOT = 4096
PORT = 8080
OUT  = "/kaggle/working/runA_out.jsonl"             # one line per finished job
FINAL = "/kaggle/working/runA_messages.jsonl"       # Arabic messages, pool format

JOBS = glob.glob("/kaggle/input/**/runA_jobs.jsonl", recursive=True)
assert JOBS, "Add the dataset that contains runA_jobs.jsonl (Add Input)."
JOBS = JOBS[0]
print("jobs file:", JOBS)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Get llama.cpp (reuse the llamacpp-t4 dataset if attached, otherwise build ~15-25 min)
import glob, os, re, shutil, subprocess

BIN = "/kaggle/working/llamacpp-bin"
LOGS = "/kaggle/working"          # logs land in the version's Output tab, even on failure
found = glob.glob("/kaggle/input/**/llama-server", recursive=True)

def run(cmd, log):
    """Run a shell command, keep the full output in a log, and show the real error if it fails."""
    with open(log, "w") as f:
        rc = subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT).returncode
    if rc != 0:
        lines = open(log, errors="replace").read().splitlines()
        hits = [i for i, l in enumerate(lines)
                if re.search(r"error|killed|undefined reference|signal 9", l, re.I)]
        if hits:
            shown = sorted({j for i in hits[:15] for j in range(i, min(i + 4, len(lines)))})
            print("\n".join(lines[j] for j in shown))
        else:
            print("\n".join(lines[-60:]))
        raise RuntimeError(f"Failed (exit {rc}): {cmd[:60]}...  Full log: {log}")

if found:
    shutil.copytree(os.path.dirname(found[0]), BIN, dirs_exist_ok=True)
    print("using prebuilt llama.cpp from", found[0])
else:
    # Newest release tag (b####) instead of master
    refs = subprocess.run("git ls-remote --tags https://github.com/ggml-org/llama.cpp",
                          shell=True, capture_output=True, text=True).stdout
    tag = "b%d" % max(int(n) for n in re.findall(r"refs/tags/b(\d+)$", refs, re.M))
    print("building llama.cpp", tag)

    shutil.rmtree("/tmp/llama.cpp", ignore_errors=True)
    run(f"git clone --depth 1 --branch {tag} https://github.com/ggml-org/llama.cpp /tmp/llama.cpp",
        f"{LOGS}/clone.log")

    # GGML_CUDA_NO_VMM=ON: Kaggle's image has no libcuda.so for CMake to link, and VMM is optional
    run("cmake -S /tmp/llama.cpp -B /tmp/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 "
        "-DGGML_CUDA_NO_VMM=ON -DBUILD_SHARED_LIBS=OFF -DLLAMA_CURL=OFF", f"{LOGS}/cmake.log")
    print("\n".join(l for l in open(f"{LOGS}/cmake.log").read().splitlines() if "CUDA" in l)[:500])

    print("compiling (silent for ~15-25 min)...", flush=True)
    run("cmake --build /tmp/llama.cpp/build --config Release -j 4 --target llama-server",
        f"{LOGS}/build.log")

    os.makedirs(BIN, exist_ok=True)
    shutil.copy("/tmp/llama.cpp/build/bin/llama-server", BIN)
    print("built. After this run, add this notebook's output as a dataset named llamacpp-t4 to skip the build next time.")

os.chmod(f"{BIN}/llama-server", 0o755)
print(subprocess.run(f"{BIN}/llama-server --version", shell=True, capture_output=True, text=True).stderr[-300:])

In [ ]:
# 3. Download the teacher model into /tmp (Kaggle's saved folder is only ~20 GB)
from huggingface_hub import hf_hub_download
t = time.time()
MODEL = hf_hub_download(MODEL_REPO, MODEL_FILE, local_dir="/tmp/models")
print(MODEL, f"{os.path.getsize(MODEL)/1e9:.2f} GB in {time.time()-t:.0f}s")

In [ ]:
# 4. Start the server: thinking OFF, split across both GPUs, PARALLEL requests at once
cmd = [f"{BIN}/llama-server", "-m", MODEL, "-ngl", "99", "--split-mode", "layer",
       "-c", str(CTX_PER_SLOT * PARALLEL), "--parallel", str(PARALLEL),
       "--jinja", "--chat-template-kwargs", '{"enable_thinking":false}',
       "--port", str(PORT)]
server = subprocess.Popen(cmd, stdout=open("/kaggle/working/server.log", "w"), stderr=subprocess.STDOUT)

import urllib.request
for i in range(180):                                  # up to 15 min to load
    try:
        if urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=5).status == 200:
            print(f"server ready after {i*5}s"); break
    except Exception:
        pass
    if server.poll() is not None:
        print(open("/kaggle/working/server.log").read()[-3000:]); raise SystemExit("server crashed")
    time.sleep(5)
else:
    raise SystemExit("server did not start; see server.log")

# Smoke test: one real request, stop here if the server rejects it or the model is still thinking
import httpx
r = httpx.post(f"http://localhost:{PORT}/v1/chat/completions", timeout=300, json={
    "messages": [{"role": "user", "content": "Translate to natural Saudi Arabic: Your parcel is on hold. Pay 2 USD at http://usps-track.co"}],
    "max_tokens": 300, "temperature": 0.4})
print("smoke test: HTTP", r.status_code)
if r.status_code != 200:
    print(r.text[:1500]); raise SystemExit("server rejected the test request")
msg = r.json()["choices"][0]["message"]
print(msg.get("content"))
if "<think>" in (msg.get("content") or "") or (msg.get("reasoning_content") or "").strip():
    raise SystemExit("model is still thinking; check --chat-template-kwargs")
print("thinking is off, OK")

In [ ]:
# 5. The client: 4 requests at a time, JSON schema enforced, every answer saved immediately
import asyncio, httpx

URL = f"http://localhost:{PORT}/v1/chat/completions"
ARABIC = re.compile(r"[؀-ۿ]")
URL_RE = re.compile(r"https?://\S+|www\.\S+|\b[\w-]+(?:\.[\w-]+)*\.(?:com|net|org|info|top|xyz|sa|co|ly|me|io|link|site|online|live|club|vip|cc|app)\b(?:/\S*)?", re.I)

def arabic_ratio(text):
    letters = [c for c in text if c.isalpha()]
    return sum(bool(ARABIC.match(c)) for c in letters) / max(1, len(letters))

def check(job, data):
    # Return (items, problems). Problems are recorded, not fatal.
    items = [data] if job["kind"] == "translate" else data.get("messages", [])
    problems = []
    for it in items:
        if arabic_ratio(it.get("text", "")) < 0.5:
            problems.append("not_arabic")
    if job["kind"] == "translate":                       # links must survive translation
        original = job["messages"][1]["content"].split("Original sender:")[0]
        for u in URL_RE.findall(original):
            if u.rstrip(".,)") not in items[0].get("text", ""):
                problems.append("url_changed"); break
    return items, problems

def load_done(*paths):
    done = set()
    for p in paths:
        for line in open(p, encoding="utf-8"):
            r = json.loads(line)
            if r.get("ok"):
                done.add(r["job_id"])
    return done

async def run(jobs, out_path=OUT, parallel=PARALLEL, url=URL, retries=3):
    previous = [p for p in glob.glob("/kaggle/input/**/runA_out.jsonl", recursive=True)]
    done = load_done(*previous, *([out_path] if os.path.exists(out_path) else []))
    # carry over answers from an attached previous run
    if previous and not os.path.exists(out_path):
        with open(out_path, "w", encoding="utf-8") as f:
            for p in previous:
                f.writelines(l for l in open(p, encoding="utf-8") if json.loads(l).get("ok"))
    todo = [j for j in jobs if j["job_id"] not in done]
    print(f"{len(done)} already done, {len(todo)} to go")
    if not todo:
        return
    sem, lock = asyncio.Semaphore(parallel), asyncio.Lock()
    stats = {"n": 0, "fail": 0, "t0": time.time()}
    out = open(out_path, "a", encoding="utf-8")

    async def one(client, job):
        async with sem:
            rec = {"job_id": job["job_id"], "ok": False}
            for attempt in range(retries):
                t = time.time()
                try:
                    body = {"model": "teacher", "messages": job["messages"],
                            "response_format": {"type": "json_schema",
                                                "json_schema": {"name": "out", "schema": job["schema"]}},
                            "chat_template_kwargs": {"enable_thinking": False},   # thinking off (Qwen3.8 template)
                            **job["sampling"]}
                    r = await client.post(url, json=body, timeout=900)
                    if r.status_code >= 400:                                      # keep the server's reason
                        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:300]}")
                    msg = r.json()["choices"][0]["message"]
                    content = msg.get("content") or ""
                    thinking = len(msg.get("reasoning_content") or "") + len(re.findall(r"<think>.+?</think>", content, flags=re.S))
                    content = re.sub(r"<think>.*?</think>", "", content, flags=re.S).strip()
                    items, problems = check(job, json.loads(content))
                    rec = {"job_id": job["job_id"], "ok": True, "items": items,
                           "problems": problems, "seconds": round(time.time() - t, 1),
                           "thinking": thinking}
                    break
                except Exception as e:
                    rec["error"] = f"{type(e).__name__}: {str(e)[:400]}"
            async with lock:
                out.write(json.dumps(rec, ensure_ascii=False) + "\n"); out.flush()
                stats["n"] += 1; stats["fail"] += not rec["ok"]
                if stats["n"] % 25 == 0 or stats["n"] == len(todo):
                    el = time.time() - stats["t0"]; rate = stats["n"] / el
                    print(f"{stats['n']}/{len(todo)}  failed={stats['fail']}  "
                          f"{rate*60:.1f} jobs/min  ETA {(len(todo)-stats['n'])/rate/60:.0f} min", flush=True)

    async with httpx.AsyncClient() as client:
        await asyncio.gather(*(one(client, j) for j in todo))
    out.close()

ALL_JOBS = [json.loads(l) for l in open(JOBS, encoding="utf-8")]
print(len(ALL_JOBS), "jobs loaded")

In [ ]:
# 6. Timing test: 20 jobs, then an estimate for the whole run
t = time.time()
await run(ALL_JOBS[:20])
per_job = (time.time() - t) / 20
print(f"~{per_job:.1f}s per job with {PARALLEL} in parallel -> whole run ~{per_job*len(ALL_JOBS)/3600:.1f} hours")
test = [json.loads(l) for l in list(open(OUT, encoding="utf-8"))[:20]]
if any(r.get("thinking") for r in test):
    print("WARNING: the model is still 'thinking' before answering, which makes the run much slower. "
          "Stop and check the --chat-template-kwargs line in step 4.")
refusals = [r for r in test if r.get("ok") and any(w in json.dumps(r["items"], ensure_ascii=False).lower()
            for w in ("i can't", "i cannot", "لا أستطيع", "لا يمكنني"))]
print(f"refusals in the test: {len(refusals)}")
for line in list(open(OUT, encoding="utf-8"))[:3]:
    r = json.loads(line); print(json.dumps(r.get("items", r), ensure_ascii=False)[:400], "\n")

In [ ]:
# 7. The full run (skips everything already done)
await run(ALL_JOBS)

In [ ]:
# 8. Turn the answers into pool records + a quality report
from collections import Counter
jobs_by_id = {j["job_id"]: j for j in ALL_JOBS}
recs, probs, seen, dups = [], Counter(), set(), 0
for line in open(OUT, encoding="utf-8"):
    r = json.loads(line)
    if not r.get("ok"):
        continue
    job = jobs_by_id[r["job_id"]]
    for p in r["problems"]: probs[p] += 1
    for i, it in enumerate(r["items"]):
        text = it.get("text", "").strip()
        key = re.sub(r"\W+", " ", text).strip()
        if not text or key in seen:
            dups += 1; continue
        seen.add(key)
        recs.append({
            "id": f'{r["job_id"]}' + ("" if job["kind"] == "translate" else f"-{i}"),
            "source": job["source"], "language": "ar",
            "sender": it.get("sender", "").strip(), "subject": "", "text": text,
            "human_label": job["label"],        # intended label
            "origin": "teacher_translate" if job["kind"] == "translate" else "teacher_generate",
            "eml_path": None, "parent_id": job["parent_id"], "scenario": job["scenario"],
            "problems": r["problems"],
        })
with open(FINAL, "w", encoding="utf-8") as f:
    for x in recs: f.write(json.dumps(x, ensure_ascii=False) + "\n")

failed = sum(1 for l in open(OUT, encoding="utf-8") if not json.loads(l).get("ok"))
print(f"{len(recs)} Arabic messages -> {FINAL}")
print("by kind/label:", dict(Counter((x['origin'], x['human_label']) for x in recs)))
print("problems:", dict(probs), "| duplicates/empty dropped:", dups, "| failed jobs:", failed)
import random
for x in random.sample(recs, min(8, len(recs))):
    print(f"\n[{x['origin']} | {x['human_label']} | {x['sender']}]\n{x['text']}")

**When it finishes:** download `runA_messages.jsonl` from the notebook's Output tab into `hather/data/out/`.
If failed jobs > 0, run the notebook again with this version's output attached; it only redoes the failures.